# DwightGPT on Colab
Runtime → Change runtime type → **T4 GPU** (or better). The whole pipeline (data, pretraining, SFT, demo) takes about 1–1.5 hours on a free T4 with the `small` preset.

In [1]:
!git clone https://github.com/Hersh96/tiny-instruct-gpt.git
%cd tiny-instruct-gpt
!pip -q install -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

Cloning into 'tiny-instruct-gpt'...
remote: Enumerating objects: 38, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 38 (delta 12), reused 38 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (38/38), 34.00 KiB | 4.86 MiB/s, done.
Resolving deltas: 100% (12/12), done.
/content/tiny-instruct-gpt
name, memory.total [MiB]
Tesla T4, 15360 MiB


## 1. Data + tokenizer
Downloads ~600 MB of TinyStories and ~150 MB of TinyStoriesInstruct, trains the BPE tokenizer and writes token files to `data/`.

In [2]:
!python scripts/prepare.py --pretrain_mb 600 --sft_mb 150 --vocab_size 8192

downloading https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-train.txt -> data/TinyStoriesV2-GPT4-train.txt.600MB
  600/600 MB
downloading https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-valid.txt -> data/TinyStoriesV2-GPT4-valid.txt
  23/23 MB
downloading https://huggingface.co/datasets/roneneldan/TinyStoriesInstruct/resolve/main/TinyStories-Instruct-train.txt -> data/TinyStories-Instruct-train.txt.150MB
  150/150 MB
downloading https://huggingface.co/datasets/roneneldan/TinyStoriesInstruct/resolve/main/TinyStories-Instruct-valid.txt -> data/TinyStories-Instruct-valid.txt
  27/27 MB
pretrain docs: 732,090 train / 27,629 val
training BPE (vocab 8192) on 30.0 MB ...
  merge 500/7932: b' pretty'
  merge 1000/7932: b' own'
  merge 1500/7932: b' shin'
  merge 2000/7932: b' fighting'
  merge 2500/7932: b' fright'
  merge 3000/7932: b' built'
  merge 3500/7932: b'ushed'
  merge 4000/7932: b' Bubbles'
  merge 4500/79

## 2. Pretraining

In [ ]:
!python train.py --stage pretrain --preset small --max_iters 5000

model: 9.44M non-embedding params, 12.59M total
config: {'vocab_size': 8192, 'block_size': 512, 'n_layer': 6, 'n_head': 6, 'n_kv_head': 2, 'n_embd': 384, 'dropout': 0.0, 'rope_theta': 10000.0, 'norm': 'rmsnorm', 'mlp': 'swiglu', 'pos': 'rope'}
compiling model with torch.compile ...
tokens per iteration (upper bound for sft): 65,536
W1125 05:31:34.680000 1675 torch/_inductor/utils.py:1806] [0/0] Not enough SMs to use max_autotune_gemm mode
/usr/local/lib/python3.13/dist-packages/torch/_inductor/compile_fx.py:3403: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
[eval] iter 0: train 9.0820, val 9.0829
/usr/local/lib/python3.13/dist-packages/torch/_inductor/compile_fx.py:3403: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/_inductor/compile_fx.py:3403: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/

In [ ]:
!python sample.py --ckpt out/pretrain/ckpt.pt --prompt "Once upon a time, there was a" --num_samples 2

## 3. Supervised fine-tuning (instruction following)

In [ ]:
!python train.py --stage sft --init_from out/pretrain/ckpt.pt --max_iters 1500

In [ ]:
!python sample.py --ckpt out/sft/ckpt.pt --chat "Write a short story about a brave little turtle. Use the words river and friend. Dwight should be in the story."

## 4. Loss curves and benchmarks

In [ ]:
!python scripts/plot.py out/pretrain/log.csv out/sft/log.csv --out assets/loss.png
!python bench.py --preset small
from IPython.display import Image, Markdown, display
display(Image('assets/loss.png'))
display(Markdown(open('assets/benchmarks.md').read()))

## 5. Optional: ablations (≈5 short runs)

In [ ]:
!ITERS=1500 bash scripts/ablate.sh

## 6. Live demo
Prints a public `gradio.live` link you can share for 72 hours.

In [ ]:
!python app.py --share

## 7. Save your work
Download the checkpoints and plots (or copy them to Google Drive).

In [ ]:
!zip -r results.zip out/pretrain/ckpt.pt out/sft/ckpt.pt out/*/log.csv data/tokenizer.json assets/
from google.colab import files
files.download('results.zip')